# 8-dars: CNN — kompyuter rasmni qanday «ko’radi»?

**Hikoya.** Telefoningizning galereyasida qidiruvga **«mushuk»** deb yozasiz. Telefon yuzlab rasm ichidan mushuklarni topib beradi. Hech kim ularga «mushuk» deb yozib qo’ymagan. Telefon rasmning **ichiga qarab** bilib oladi.

7-darsda tarmoq oq-qora **28 × 28** raqam rasmini taniy oldi. Bugun **rangli** rasmlar: mashina, kema, it, mushuk, samolyot… Va bitta yangi muammo: **oddiy tarmoq rasmda yomon ishlaydi.** Nega? Va uni nima bilan almashtiramiz? Javob: **CNN** (*konvolyutsion neyron tarmoq*), ya’ni rasm uchun yasalgan tarmoq.

**Bugun siz:**
- **nega** 7-darsdagi tarmoq rasmga yaxshi mos kelmasligini ko’rasiz;
- **filtr** (naqsh qidiruvchi) nima ekanini qo’lda hisoblab ko’rasiz: 2-darsdagi filtrlarning «o’rganadigan» akasi;
- **pooling** va **kanal** nima ekanini tushunasiz;
- o’zingizning **CNN** ingizni yozib, 7-darsdagi tarmoq bilan **solishtirasiz**;
- CNN ning ichiga qarab, uning filtrlari **nimani izlayotganini** ko’rasiz;
- **augmentation** (rasmni aylantirish, kesish) nima uchun kerakligini ko’rasiz.

**Asosiy fikr:** rasmda **qo’shni piksellar bir-biri bilan bog’liq**. CNN shu bog’liqlikdan foydalanadi, oddiy tarmoq esa piksellarni bir qatorga yoyib, bu bog’liqlikni **yo’qotadi**.

**Notebook qanday ishlaydi:** matn kataklari — tushuntirish; kod kataklari ▶ (yoki Shift + Enter) bilan ishga tushadi. Kataklarni yuqoridan pastga, tartib bilan ishga tushiring.

⚠️ Avval: **Copy to Drive** (yoki **File → Save a copy in Drive**). Eng yaxshisi: **Runtime → Change runtime type → T4 GPU** (o’qitish bir necha marta tezlashadi).

## Bugungi so’zlar (shu jadvalga qaytib turing)

| So’z | Oddiy tilda | Misol |
|---|---|---|
| **CNN** | rasm uchun tarmoq: filtrlar bilan rasmni «ko’radi» | telefon galereyasi |
| **filtr** (kernel) | kichik jadval (3 × 3): rasm ustida suriladigan «naqsh qidiruvchi» | vertikal chiziq qidiruvchi |
| **konvolyutsiya** | filtrni rasm ustida surib, har joyda dot product hisoblash | 7-darsdagi `(x * w).sum()` ni har joyda |
| **kanal** | rasmning «qatlami»: rangli rasmda 3 ta (qizil, yashil, ko’k) | `[3, 32, 32]` |
| **xarita** (feature map) | bitta filtr rasmdan topgan narsalar jadvali | «chiziqlar xaritasi» |
| **pooling** | xaritani kichraytirish: har 2 × 2 dan eng kattasini olish | 4 o’quvchidan eng balandi |
| **augmentation** | rasmni biroz o’zgartirib, ma’lumotni ko’paytirish | oynadagi aks, kesilgan rasm |
| **CIFAR-10** | 60 000 ta kichik rangli rasm, 10 xil narsa | mashina, kema, it… |

## 0. Kutubxonalar

7-darsdagi kutubxonalar. Yangi narsa yo’q: faqat `numpy` (2-dars) qaytib keldi.

In [ ]:
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

qurilma = "cuda" if torch.cuda.is_available() else "cpu"      # 7-dars: GPU bormi?
print("Qurilma:", qurilma)

## 1. CIFAR-10: rangli rasmlar

**Hayotda:** telefon galereyasi, avtomobil kamerasi, Google Lens — hammasi rangli rasm bilan ishlaydi.

**CIFAR-10** — 60 000 ta **kichik rangli rasm** (32 × 32 piksel), 10 xil narsa. MNIST kabi, lekin ancha **qiyin**: raqam oq-qora va markazda edi, bu yerda esa rang, fon, turli burchak.

- **train: 50 000** (o’rganish uchun), **test: 10 000** (imtihon, tarmoq ko’rmagan rasmlar). 7-darsdagidek.

Yuklaymiz (birinchi marta bir necha soniya):

In [ ]:
sinflar = ["samolyot", "mashina", "qush", "mushuk", "kiyik", "it", "qurbaqa", "ot", "kema", "yuk mashinasi"]

aylantir = transforms.ToTensor()           # rasm -> tensor, piksel 0..1
train_data = datasets.CIFAR10("data", train=True, download=True, transform=aylantir)
test_data = datasets.CIFAR10("data", train=False, download=True, transform=aylantir)

print("train:", len(train_data), " test:", len(test_data))

**Bitta rasmni** ochamiz: u qanday jadval?

In [ ]:
rasm, javob = train_data[0]

print("javob:", sinflar[javob])
print("rasm o'lchami:", rasm.shape)       # [3, 32, 32]: 3 kanal, 32 qator, 32 ustun

O’lcham **`[3, 32, 32]`**: **3 kanal** (qizil, yashil, ko’k: 2-darsdagidek), 32 qator, 32 ustun. Demak, bitta rasm — 3 × 32 × 32 = **3072 ta son**. 7-darsdagi raqam rasmida 784 ta edi.

10 ta rasmni javoblari bilan ko’ramiz:

In [ ]:
plt.figure(figsize=(12, 2.8))
for i in range(10):
    r, j = train_data[i]
    plt.subplot(1, 10, i + 1)
    plt.imshow(r.permute(1, 2, 0))        # [3,32,32] -> [32,32,3]: matplotlib shuni kutadi
    plt.title(sinflar[j], fontsize=9)
    plt.axis("off")
plt.show()

Rasmlar **kichik va xira** (32 × 32), ba’zan odam ham ikkilanadi. Aynan shuning uchun CIFAR-10 mashhur: oson emas.

## 2. Nega oddiy tarmoq rasmda yomon ishlaydi?

**Hayotda:** mushukning rasmini **ikki piksel o’ngga** surib qo’ysangiz, u baribir mushuk. Siz buni **bir qarashda** bilasiz. Oddiy tarmoq esa yo’q. Ko’ramiz.

7-darsda tarmoq rasmni **bir qatorga yoyib** (`Flatten`) oldi: 3072 ta son ketma-ket. Ikkita muammo:

**Muammo 1: surilgan rasm — boshqa sonlar.** Rasmni 2 piksel o’ngga suramiz (`torch.roll`). Odam uchun u o’sha rasm. Sonlar bo’yicha esa qanchalik farq qiladi?

In [ ]:
rasm, _ = train_data[0]
surilgan = torch.roll(rasm, shifts=2, dims=2)        # 2 piksel o'ngga

plt.figure(figsize=(5, 2.6))
plt.subplot(1, 2, 1); plt.imshow(rasm.permute(1, 2, 0)); plt.title("asl"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(surilgan.permute(1, 2, 0)); plt.title("2 piksel o'ngda"); plt.axis("off")
plt.show()

Ikki rasm bir xil, faqat surilgan. Piksellar bo’yicha **o’rtacha farqni** hisoblaymiz (0 — bir xil, katta — juda boshqa). Solishtirish uchun **boshqa mashhur rasm** bilan ham:

In [ ]:
def farq(a, b):
    return (a - b).abs().mean().item()          # piksellarning o'rtacha farqi

surish_farqi = []
boshqa_farqi = []
for i in range(500):
    a, _ = train_data[i]
    b, _ = train_data[i + 500]
    surish_farqi.append(farq(a, torch.roll(a, shifts=2, dims=2)))     # o'zi va surilgani
    boshqa_farqi.append(farq(a, b))                                     # ikki turli rasm

print(f"o'zi va 2 piksel surilgani: o'rtacha farq {np.mean(surish_farqi):.3f}")
print(f"ikki turli rasm:            o'rtacha farq {np.mean(boshqa_farqi):.3f}")

**Natija:** 2 piksel surilgan **o’sha rasm** bilan farq taxminan **0.09**, ikki **turli rasm** orasida esa taxminan **0.29**. Ya’ni surilgan nusxa, odam uchun **aynan o’sha rasm** bo’lsa ham, sonlar bo’yicha turli rasmning **uchdan biri** qadar uzoqlashdi. Oddiy tarmoq piksellarni **joyi bilan** solishtiradi, shuning uchun surilgan rasmni u **yangi, tanish bo’lmagan** rasm deb ko’radi.

**Muammo 2: qo’shni piksellar bilimi yo’qoladi.** Oddiy tarmoq 3072 ta sonni **bir qator** deb oladi va qaysi piksel qaysining yonida turganini **bilmaydi**. Qo’shni piksellarni boshqa joyga **aralashtirib** yuborsak ham tarmoq uchun **hech narsa o’zgarmaydi** (hamma rasm bir xil tartibda aralashtirilsa). Odam uchun esa rasm butunlay buziladi:

In [ ]:
torch.manual_seed(0)
tartib = torch.randperm(32 * 32)                       # tasodifiy tartib (hamma rasmga bir xil qo'llanadi)

aralash = rasm.reshape(3, -1)[:, tartib].reshape(3, 32, 32)

plt.figure(figsize=(5, 2.6))
plt.subplot(1, 2, 1); plt.imshow(rasm.permute(1, 2, 0)); plt.title("asl"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(aralash.permute(1, 2, 0)); plt.title("piksellar aralashgan"); plt.axis("off")
plt.show()

O’ngdagi rasmni **siz** taniy olmaysiz. Oddiy tarmoqqa esa ikkalasi **teng**: u pikselning **yonida nima turganini** ko’rmaydi.

**Xulosa.** Rasmda **qo’shni piksellar** muhim (chiziq, burchak, ko’z, quloq — hammasi qo’shni piksellardan hosil bo’ladi) va **naqsh rasmning istalgan joyida** bo’lishi mumkin. Bizga **kichik «naqsh qidiruvchi»** kerak: u rasm ustida suriladi va har joyda naqsh bor-yo’qligini tekshiradi. Bu **filtr**.

## 3. Filtr: naqsh qidiruvchi

**Hayotda:** siz kitobda **bitta harfni** izlayotganingizda sahifani **qarab chiqasiz**: ko’zingiz har joyda «bu o’sha harfmi?» deb tekshiradi. Filtr ham shunday, faqat u **sonlar** bilan ishlaydi.

Filtr — kichik jadval (masalan 3 × 3). Uni rasm ustiga qo’yib, **piksellarni filtr sonlariga ko’paytirib qo’shamiz**. Bu — 7-darsdagi `(x * w).sum()` (dot product)ning aynan o’zi, faqat jadval bilan.

**Avval juda kichik misol.** 6 × 6 «rasm»: chap yarmi qora (0), o’ng yarmi oq (9). Ularning o’rtasida **vertikal chegara**.

In [ ]:
rasm6 = np.array([[0, 0, 0, 9, 9, 9],
                  [0, 0, 0, 9, 9, 9],
                  [0, 0, 0, 9, 9, 9],
                  [0, 0, 0, 9, 9, 9],
                  [0, 0, 0, 9, 9, 9],
                  [0, 0, 0, 9, 9, 9]])

plt.figure(figsize=(2.6, 2.6))
plt.imshow(rasm6, cmap="gray"); plt.axis("off"); plt.title("6 x 6 rasm")
plt.show()

Endi **vertikal chegarani topadigan filtr**: chap ustuni **−1**, o’rtasi **0**, o’ng ustuni **+1**. Ma’nosi: «chapda qora, o’ngda oq bo’lsa — katta son chiqsin».

In [ ]:
filtr = np.array([[-1, 0, 1],
                  [-1, 0, 1],
                  [-1, 0, 1]])

print(filtr)

**Bitta joyda hisoblaymiz.** Filtrni rasmning yuqori chap qismidan **1 ustun o’ngda** (chegara yaqinida) turgan 3 × 3 «oynaga» qo’yamiz:

In [ ]:
oyna = rasm6[0:3, 1:4]               # 0-2 qatorlar, 1-3 ustunlar: 3 x 3 bo'lak

print("oyna:")
print(oyna)
print("oyna * filtr (juft-juft ko'paytma):")
print(oyna * filtr)
print("yig'indi:", (oyna * filtr).sum())

Hisob: oynada chap ustun 0, o’rta 0, o’ng ustun 9 (uch qatorda). Filtrda o’ng ustun +1: `9 + 9 + 9 = 27`. **27 — katta son: «bu yerda chegara bor!»**

Chegara bo’lmagan joyda nima chiqadi? Oynani **chapga**, hammasi qora joyga qo’yamiz:

In [ ]:
oyna_qora = rasm6[0:3, 0:3]          # hammasi 0
print((oyna_qora * filtr).sum())     # chegara yo'q -> 0

oyna_oq = rasm6[0:3, 3:6]            # hammasi 9
print((oyna_oq * filtr).sum())       # chap -9*3 + o'ng +9*3 = 0 (chegara yo'q)

Ikkalasida ham **0**: tekis joyda (hammasi qora yoki hammasi oq) filtr jim turadi. **Faqat chegarada** katta son beradi. Mana, filtrning ishi: **«naqsh bor joyda katta son»**.

**Mustaqil ish 1.** Filtrni **butun rasm ustida** suring: har joyda yuqoridagi hisobni takrorlang. Natija — **xarita** (u yerda qayerda naqsh borligi). 3 × 3 filtr 6 × 6 rasmda **4 × 4** joyga sig’adi (6 − 3 + 1 = 4).

In [ ]:
# TODO: konv funksiyasini to'ldiring
def konv(rasm, filtr):
    n = rasm.shape[0] - filtr.shape[0] + 1                    # natija o'lchami
    natija = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            oyna = rasm[i:i + 3, j:j + 3]                      # (i, j) dan boshlanuvchi 3 x 3 oyna
            natija[i, j] = 0                                   # TODO: (oyna * filtr).sum() yozing
    return natija

xarita = konv(rasm6, filtr)
print(xarita)
assert xarita.tolist()[0] == [0, 27, 27, 0], f"birinchi qator {xarita.tolist()[0]}, lekin [0, 27, 27, 0] bo'lishi kerak. natija[i, j] = (oyna * filtr).sum() yozdingizmi?"

Xarita: har qatorda **`[0, 27, 27, 0]`**. Katta sonlar o’rtada, aynan chegara turgan **joyda**. Filtr chegarani **topdi**, qaysi qatorda bo’lishidan qat’i nazar. Shuncha.

**Muhim:** filtr **bir xil** (9 ta son) va u rasmning **hamma** joyiga qo’llanadi. Naqsh rasmning qayerida bo’lsa ham topiladi, bu **surilish muammosini** hal qiladi.

**Sinab ko’ring.** Filtrni **burib** qo’ysak (`filtr.T` — satr va ustunlarni almashtirish), u **gorizontal** chegarani topadi. Avval taxmin qiling: `konv(rasm6, filtr.T)` bizning rasmda nima beradi (rasmda faqat vertikal chegara bor)?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

Hammasi **0**: gorizontal chegara yo’q, **shuning uchun** gorizontal filtr jim. Filtr faqat **o’z naqshini** topadi.

**Haqiqiy rasmda sinab ko’ramiz.** CIFAR rasmini oq-qora qilamiz (3 kanal o’rtachasi: 2-darsdagi oq-qora kabi) va 3 ta filtr qo’llaymiz: **vertikal**, **gorizontal** chegara va **xiralashtirish**.

In [ ]:
oq_qora = train_data[4][0].mean(dim=0).numpy()            # [32, 32] oq-qora rasm (3 kanal o'rtachasi)

vertikal = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
gorizontal = vertikal.T
xira = np.ones((3, 3)) / 9                                 # o'rtacha: xiralashtiradi

plt.figure(figsize=(11, 3))
for k, (nom, f) in enumerate([("asl", None), ("vertikal chegara", vertikal), ("gorizontal chegara", gorizontal), ("xira", xira)]):
    plt.subplot(1, 4, k + 1)
    plt.imshow(oq_qora if f is None else konv(oq_qora, f), cmap="gray")
    plt.title(nom); plt.axis("off")
plt.show()

- **Vertikal filtr:** vertikal chiziqlar (ustun, devor, chetlar) **yorqin**, boshqalar qora;
- **Gorizontal filtr:** gorizontal chiziqlar (gorizont, tom) yorqin;
- **Xira filtr:** hamma 9 ta son bir xil → o’rtacha → rasm **xira** bo’ldi.

Bu 2-darsdagi **filtrlar** (oq-qora, yorqinlik, xira, qirra)ning aynan o’zi, faqat endi filtrni **oyna bo’ylab suramiz**.

**Eng muhim fikr:** bu filtrlarni **biz qo’lda** yozdik (−1, 0, 1). **CNN filtr sonlarini o’zi o’rganadi**, xuddi 7-darsda tarmoq og’irliklarni o’rganganidek! Biz «vertikal chegara qidir» deb aytmaymiz: tarmoq qaysi naqshlar foydali ekanini **o’zi** topadi.

## 4. PyTorch: `nn.Conv2d`

**Hayotda:** o’zingiz siklni yozmaysiz: kalkulyator tayyor. `nn.Conv2d` = bizning `konv` funksiyamiz, faqat **tez** va **o’rganadigan**.

Avval bitta filtr: `nn.Conv2d(1, 1, 3)` degani: **1 kirish kanal → 1 chiqish kanal, 3 × 3 filtr**. Unga o’zimizning vertikal filtrimizni qo’yamiz va natija bir xil chiqishini tekshiramiz:

In [ ]:
conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)

with torch.no_grad():
    conv.weight.copy_(torch.tensor(vertikal, dtype=torch.float32).reshape(1, 1, 3, 3))   # filtrni qo'ydik

x = torch.tensor(rasm6, dtype=torch.float32).reshape(1, 1, 6, 6)       # [batch, kanal, qator, ustun]
print(conv(x)[0, 0].detach())

**O’sha `[0, 27, 27, 0]`** qatorlari. `nn.Conv2d` bizning `konv` ning aynan o’zi.

Endi filtr sonlarini **qo’ymaymiz**: ular boshida **tasodifiy**, o’qitish topadi (7-dardagi `nn.Linear` og’irliklari kabi). Bir yo’la **ko’p filtr**: `nn.Conv2d(3, 16, 3)` — **3 kanal** (rangli rasm) kiradi, **16 ta filtr** ishlaydi, **16 ta xarita** chiqadi.

In [ ]:
conv16 = nn.Conv2d(3, 16, kernel_size=3)

print("filtrlar shakli:", tuple(conv16.weight.shape))        # [16, 3, 3, 3]: 16 ta filtr, har biri 3 kanal x 3 x 3
print("sonlar soni:", sum(p.numel() for p in conv16.parameters()))     # 16*3*3*3 + 16

x = rasm.unsqueeze(0)                                        # [1, 3, 32, 32]: 1 ta rasm
print("kirish:", tuple(x.shape), "-> chiqish:", tuple(conv16(x).shape))

- **filtrlar `[16, 3, 3, 3]`**: 16 ta filtr, har biri **3 kanal × 3 × 3** (rangning har kanali uchun alohida 3 × 3 jadval);
- **sonlar: 448** (16 · 27 + 16 ta `b`). **16 ta filtr uchun atigi 448 ta son!**;
- **chiqish `[1, 16, 30, 30]`**: 16 ta xarita, har biri 30 × 30 (32 − 3 + 1 = 30: chetlarda filtr sig’maydi).

**Sinab ko’ring.** Chetlar yo’qolmasin desak, rasm atrofiga **nol qatori** (`padding=1`) qo’shiladi. Avval taxmin qiling: `nn.Conv2d(3, 16, 3, padding=1)` chiqishi qanday shaklda?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 5. ReLU va pooling: xaritani kichraytiramiz

**Hayotda:** katta guruh rasmidan **har 4 kishidan eng balandini** tanlab, kichik «vakillar» rasmini yasaysiz. Rasm kichrayadi, **asosiy narsa** qoladi. Bu pooling.

Filtrdan keyin ikkita oddiy qadam (har CNN da bor):

1. **ReLU** (6-dars): manfiy sonlarni 0 qiladi. «Naqsh topilmadi» = jim.
2. **Pooling**: xaritani **2 marta kichraytirish**: har **2 × 2** bo’lakdan **eng kattasini** olamiz (**max pooling**). Naqsh qayerda ekani **taxminan** saqlanadi, lekin tarmoq tez ishlaydi.

**Kichik misol.** 4 × 4 xarita → 2 × 2:

In [ ]:
xarita4 = np.array([[1, 3, 2, 0],
                    [4, 2, 1, 5],
                    [0, 1, 7, 3],
                    [2, 6, 4, 8]])

print(xarita4)
print("birinchi 2x2 bo'lak:", xarita4[0:2, 0:2].tolist(), "-> eng katta:", xarita4[0:2, 0:2].max())

Birinchi bo’lak `[[1, 3], [4, 2]]`: eng kattasi **4**. Hamma 4 ta bo’lak uchun shunday qilamiz. 1-darsdagi `reshape(8, 4, 8, 4).sum(axis=(1, 3))` ni eslang: u rasmni bo’laklarga bo’lib **qo’shgan** edi. Bu yerda esa **eng kattasini** olamiz.

**Mustaqil ish 2.** `maks_pool(x)` funksiyasini yozing (kvadrat xarita uchun): rasmni `(n/2, 2, n/2, 2)` shaklga keltirib, 2- va 4-o’qlar bo’yicha eng kattasini oling.

In [ ]:
# TODO: maks_pool funksiyasini to'ldiring
def maks_pool(x):
    n = x.shape[0]
    return x           # TODO: x.reshape(n // 2, 2, n // 2, 2).max(axis=(1, 3)) yozing

natija2 = maks_pool(xarita4)
print(natija2)
assert natija2.tolist() == [[4, 5], [6, 8]], f"natija {natija2.tolist()}, lekin [[4, 5], [6, 8]] bo'lishi kerak. reshape(n // 2, 2, n // 2, 2).max(axis=(1, 3)) yozdingizmi?"

**`[[4, 5], [6, 8]]`**: 4 ta bo’lakning eng kattalari. Xarita **4 × 4 dan 2 × 2 ga** (4 marta kam son). PyTorch’da: `nn.MaxPool2d(2)`.

Endi **bitta blok**: **filtr → ReLU → pooling**. Haqiqiy rasm bilan shakllar qanday o’zgaradi?

In [ ]:
blok = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1),     # 16 ta filtr: [1, 16, 32, 32]
    nn.ReLU(),                                       # manfiyni 0 qiladi
    nn.MaxPool2d(2),                                 # 2 marta kichraytiradi: [1, 16, 16, 16]
)

x = rasm.unsqueeze(0)
print("kirish:", tuple(x.shape))
for qatlam in blok:
    x = qatlam(x)
    print(f"{type(qatlam).__name__:<10} -> {tuple(x.shape)}")

Rasm **3 × 32 × 32** (3072 son) → **16 × 16 × 16** (4096 son): xaritalar soni **oshdi** (3 dan 16 ga), o’lcham **kichraydi** (32 dan 16 ga). Blokdan blokka xaritalar **ko’payadi**, o’lcham **kichrayadi**: tarmoq rasmni tobora **umumiy** ko’radi.

## 6. CNN: bloklarni zanjirga ulaymiz

**Hayotda:** siz rasmga avval **mayda** narsalarga qaraysiz (chiziq, nuqta), keyin ularni birlashtirib **kattaroq** narsani ko’rasiz (ko’z, quloq), keyin **butun** hayvonni tanyasiz. CNN ham shunday: **birinchi blok** — chiziqlar, **ikkinchi blok** — ularning birikmalari, oxirgi qadam — qaror.

Bizning CNN:

| Qadam | Nima qiladi | Shakl |
|---|---|---|
| kirish | rangli rasm | `[3, 32, 32]` |
| **1-blok**: Conv(3 → 16) + ReLU + Pool | chiziq va ranglarni topadi | `[16, 16, 16]` |
| **2-blok**: Conv(16 → 32) + ReLU + Pool | chiziqlardan shakllar yasaydi | `[32, 8, 8]` |
| `Flatten` | jadvalni qatorga yoyadi | `[2048]` |
| `Linear(2048 → 10)` | 10 ta sinf uchun ball | `[10]` |

2-blok 16 ta xaritadan **32 ta yangi xarita** yasaydi (har yangi filtr 16 ta xaritani birga ko’radi). `Flatten` dan keyin: 32 · 8 · 8 = **2048** son, ulardan 10 ta ball.

**Mustaqil ish 3.** CNN ni yasang. 1-blok tayyor, **2-blok** va **oxirgi qadam** (Flatten + Linear) sizdan.

In [ ]:
# TODO: cnn_yasa funksiyasini to'ldiring
def cnn_yasa():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),       # 1-blok: [3,32,32] -> [16,16,16]
        # TODO: 2-blok: nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)    -> [32, 8, 8]
        # TODO: nn.Flatten(), nn.Linear(32 * 8 * 8, 10)                                  -> [10]
    )

model_cnn = cnn_yasa()
soni = sum(p.numel() for p in model_cnn.parameters())
assert soni == 25578, f"modelda {soni} ta son bor, lekin 25578 bo'lishi kerak. 2-blok va Flatten + Linear(32 * 8 * 8, 10) ni qo'shdingizmi?"
print("CNN dagi sonlar:", soni)

**25 578 ta son**. Hammasi to’g’ri ulangani uchun rasm bir uchidan kirib, 10 ta ballga aylanadi. Shakllar qanday o’zgarishini qadam-baqadam ko’ramiz:

In [ ]:
x = rasm.unsqueeze(0)
print(f"{'kirish':<10} -> {tuple(x.shape)}")
for qatlam in model_cnn:
    x = qatlam(x)
    print(f"{type(qatlam).__name__:<10} -> {tuple(x.shape)}")

Shakl **`[1, 3, 32, 32]`** dan **`[1, 10]`** gacha: har qatlam o’lchamni o’zgartirib boradi. Oxirida **10 ta ball**, qaysi biri baland bo’lsa, shu sinf (7-dars: `argmax`).

7-darsdagi **oddiy tarmoq** (MLP) bilan **sonlar sonini** solishtiramiz. U 3072 ta sonni to’g’ri 128 neyronga beradi:

In [ ]:
def mlp_yasa():
    return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 128), nn.ReLU(), nn.Linear(128, 10))

model_mlp = mlp_yasa()
mlp_soni = sum(p.numel() for p in model_mlp.parameters())
cnn_soni = sum(p.numel() for p in model_cnn.parameters())
print(f"MLP: {mlp_soni:,} ta son")
print(f"CNN: {cnn_soni:,} ta son")
print(f"MLP CNN dan {mlp_soni / cnn_soni:.0f} marta katta")

CNN **~15 marta kichik**. Chunki **filtr** (9 ta son) rasmning **hamma** joyida qayta ishlatiladi. Oddiy tarmoq esa **har piksel uchun** alohida og’irlik oladi. Kam son, **ko’p** foyda. Natijani ko’ramiz.

## 7. O’qitamiz: MLP va CNN poygasi

**Hayotda:** ikki o’quvchi bir xil kitobdan o’qiydi. Biri har sahifani yodlaydi (MLP), ikkinchisi **qoidalarni** tushunadi (CNN). Imtihonda kim yaxshiroq?

7-darsdagi o’qitish tsikli, **bir xil** (oldinga → loss → nolga → orqaga → qadam). Faqat bitta qo’shimcha: `yuklagich` (qaysi ma’lumotdan o’qitish), uy ishida kerak bo’ladi. Tayyor, ko’chirib qo’ydik:

In [ ]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)


def aniqlik(model, loader):
    model.eval()
    togri, jami = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(qurilma), y.to(qurilma)
            togri += (model(x).argmax(dim=1) == y).sum().item()
            jami += len(y)
    return togri / jami


def o_qit(model, yuklagich, epochlar=3, lr=0.001):
    model = model.to(qurilma)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    natijalar = []
    for epoxa in range(epochlar):
        model.train()
        jami_loss = 0.0
        for x, y in yuklagich:
            x, y = x.to(qurilma), y.to(qurilma)
            loss = loss_fn(model(x), y)           # 1. oldinga, 2. loss
            optimizer.zero_grad()                 # 3. eski ta'sirlarni o'chir
            loss.backward()                       # 4. orqaga
            optimizer.step()                      # 5. qadam
            jami_loss += loss.item()
        aniq = aniqlik(model, test_loader)
        natijalar.append((jami_loss / len(yuklagich), aniq))
        print(f"{epoxa + 1}-epoxa: loss = {natijalar[-1][0]:.4f}, test aniqligi = {aniq * 100:.2f}%")
    return natijalar

Bu 7-darsdagi `o_qit` bilan **bir xil**. Avval **MLP** (3 epoxa):

In [ ]:
torch.manual_seed(1)
model_mlp = mlp_yasa()
t0 = time.time()
mlp_natija = o_qit(model_mlp, train_loader, epochlar=3)
print(f"vaqt: {time.time() - t0:.0f} soniya")

MLP **MNIST da 97%** bergan edi. Rangli rasmda esa taxminan **42%**: odamga oson bo’lgan masala unga qiyin (tasodif — 10%). Endi **CNN** (xuddi shu 3 epoxa, xuddi shu ma’lumot):

In [ ]:
torch.manual_seed(1)
model_cnn = cnn_yasa()
t0 = time.time()
cnn_natija = o_qit(model_cnn, train_loader, epochlar=3)
print(f"vaqt: {time.time() - t0:.0f} soniya")

CNN taxminan **58%**: MLP dan **16 foiz punkt aniqroq** va **15 marta kam son** bilan. Taqqoslash grafigi (dars natijasi):

In [ ]:
mlp_aniq = mlp_natija[-1][1] * 100
cnn_aniq = cnn_natija[-1][1] * 100

plt.figure(figsize=(9, 3.5))
plt.subplot(1, 2, 1)
plt.bar(["MLP", "CNN"], [mlp_aniq, cnn_aniq], color=["#94a3b8", "#00D1D1"])
plt.ylabel("test aniqligi, %"); plt.title("Aniqlik: CNN yaxshiroq")
for i, v in enumerate([mlp_aniq, cnn_aniq]):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center")

plt.subplot(1, 2, 2)
plt.bar(["MLP", "CNN"], [mlp_soni, cnn_soni], color=["#94a3b8", "#00D1D1"])
plt.ylabel("sonlar soni"); plt.title("Sonlar: CNN ancha kam")
plt.tight_layout()
plt.show()

Chapda **aniqlik** (CNN baland), o’ngda **sonlar soni** (CNN past). **Yaxshiroq natija, ancha kichik tarmoq.** Rasm uchun yasalgan tarmoq mana shunday ishlaydi. (58% — 3 epoxa uchun; uy ishida 5 epoxada ~62% ko’rasiz. Rangli, murakkab rasmlarda katta tarmoq va uzoq o’qitish kerak.)

**Sinab ko’ring.** Tarmoq kattaroq bo’lsa-chi? `Conv2d(3, 16, …)` o’rniga **32** filtr, ikkinchi blokda **64** filtr qo’ysak, aniqlik oshadimi? Avval taxmin qiling. (GPU bo’lmasa, bu katak bir necha daqiqa oladi: 2 epoxa.)

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 8. CNN ichiga qaraymiz: u nimani «ko’radi»?

**Hayotda:** o’quvchi tushunganini bilish uchun **«qanday yechding?»** deb so’raysiz. Biz ham tarmoqdan so’raymiz: **qaysi naqshlarni qidirishni o’rgandi?**

**Birinchi blokdagi 16 ta filtr.** Ular boshida **tasodifiy** edi. O’qitishdan keyin ular **o’rgangan** naqshlar. Har bir filtr — `3 × 3` jadval, 3 kanal (rang), shuning uchun uni **kichik rangli rasm** sifatida chizish mumkin:

In [ ]:
filtrlar = model_cnn[0].weight.detach().cpu()           # [16, 3, 3, 3]
filtrlar = (filtrlar - filtrlar.min()) / (filtrlar.max() - filtrlar.min())     # 0..1 oralig'iga keltiramiz

plt.figure(figsize=(12, 1.8))
for i in range(16):
    plt.subplot(1, 16, i + 1)
    plt.imshow(filtrlar[i].permute(1, 2, 0), interpolation="nearest")
    plt.axis("off")
plt.suptitle("Tarmoq o'rgangan 16 ta filtr (3 x 3, rangli)")
plt.show()

Bu kichik rangli kataklar — tarmoq **o’zi** topgan naqshlar: ba’zilari **yo’nalishli chiziq** (yorug’dan qorong’iga), ba’zilari **rang** (qizil, ko’k bo’lak) qidiradi. Hech kim ularga «chiziq qidir» demagan.

Endi **bitta rasm** shu 16 ta filtrdan o’tganda **xaritalar** qanday chiqadi? Har xarita — «bu filtr naqshini rasmning qayerida topdi»:

In [ ]:
rasm_k, javob_k = test_data[0]
with torch.no_grad():
    xaritalar = torch.relu(model_cnn[0](rasm_k.unsqueeze(0).to(qurilma)))[0].cpu()      # [16, 32, 32]

plt.figure(figsize=(12, 3.4))
plt.subplot(2, 9, 1); plt.imshow(rasm_k.permute(1, 2, 0)); plt.title(sinflar[javob_k], fontsize=9); plt.axis("off")
for i in range(16):
    plt.subplot(2, 9, i + 2)
    plt.imshow(xaritalar[i], cmap="gray"); plt.title(f"filtr {i + 1}", fontsize=8); plt.axis("off")
plt.show()

Chapda asl rasm, o’ngda 16 ta **xarita**. Har xaritada **yorqin joy** = shu filtr o’z naqshini **topgan** joy. Biri **chegara**, biri **fon**, biri **rang** bo’lagi: shunday qilib tarmoq rasmni **qismlarga** ajratadi.

**Chuqur qatlamlar.** Birinchi blok oddiy narsalarni (chiziq, rang) topadi. **Ikkinchi blok** bu xaritalardan **murakkabroq** shakllarni yasaydi (burchak, doira). Katta CNN larda keyingi qatlamlar **ko’z, g’ildirak, qanot** kabi qismlarni, eng oxirgisi esa **butun narsani** taniydi. Shuning uchun uni **chuqur** o’rganish (deep learning) deyishadi.

## 9. CNN Explainer: ichiga jonli qarash

Brauzerda ochiq sayt: **https://poloclub.github.io/cnn-explainer/**. U yerda CNN tarkibini **jonli** ko’rasiz: rasm tanlaysiz va u qatlamdan qatlamga qanday o’tishini kuzatasiz.

Vazifalar (har biri 1–2 daqiqa):
1. Saytda **mushuk** (tiger cat) yoki boshqa rasmni tanlang. **Birinchi qatlamda** xaritalar rasmga qanchalik o’xshashini qarang.
2. **ReLU** qatlamiga o’ting: manfiy joylar **qora** bo’lib, «jim» qolishini ko’ring.
3. **Pooling** (maxpool) qatlamida: xaritalar **2 marta kichrayganini** va eng yorqin joylar saqlanganini ko’ring.
4. Biror **xaritaga** sichqonchani olib boring: **filtr** va u rasmning qaysi bo’lagini ko’rayotgani ko’rsatiladi. Bu bizning `oyna * filtr` hisobimiz.
5. **Oxirgi qatlam**: 10 ta chiqish. Qaysi biri baland bo’lsa, shu tarmoq javobi (bizdagi `argmax`).

## 10. Augmentation: rasmni o’zgartirib ma’lumotni ko’paytirish

**Hayotda:** yangi o’yinda bir xil darvozani **har xil burchakdan** tepish orqali o’rganasiz. Tarmoq ham: bir xil rasmni **biroz o’zgartirib** ko’rsak, u rasmning **mohiyatini** o’rganadi, yodlamaydi.

**Augmentation** — o’qitish paytida har rasmni **tasodifiy o’zgartirish**:
- **ko’zgu** (chap-o’ng almashtirish): mushuk chapga qarasa ham, o’ngga qarasa ham mushuk;
- **kesish** (rasm chetidan biroz kesib, yana 32 × 32 ga keltirish): mushuk rasmning boshqa joyida;
- **yorqinlik**: qorong’i yoki yorug’ kunda olingan rasm.

Bu **yangi ma’lumot yig’maydi**, mavjud 50 000 tadan **ko’p xil** variant yasaydi. **Ogohlantirish:** augmentation har doim ham darhol yordam bermaydi: qisqa o’qitishda u hatto aniqlikni **pasaytirishi** mumkin, chunki tarmoqqa qiyinroq masala beriladi. Uy ishida buni o’zingiz o’lchaysiz.

**Mustaqil ish 4.** `augment` ni yozing: tasodifiy **ko’zgu** va tasodifiy **kesish** (`padding=4`) qo’shing.

In [ ]:
# TODO: augment ni to'ldiring
augment = transforms.Compose([
    # TODO: transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4) ni qo'shing
    transforms.ColorJitter(brightness=0.3),            # yorqinlik: tasodifiy +-30%
    transforms.ToTensor(),
])

nomlar = [type(t).__name__ for t in augment.transforms]
print(nomlar)
assert "RandomHorizontalFlip" in nomlar and "RandomCrop" in nomlar, f"augment ichida {nomlar}. RandomHorizontalFlip() va RandomCrop(32, padding=4) ni qo'shdingizmi?"

Bitta rasmni **8 marta** o’zgartirib ko’ramiz. Har safar boshqacha, lekin hammasi **o’sha mushuk/it/mashina**:

In [ ]:
pil_data = datasets.CIFAR10("data", train=True, download=True)        # transform yo'q: asl (PIL) rasm
pil_rasm, pil_javob = pil_data[7]

plt.figure(figsize=(12, 2.4))
plt.subplot(1, 9, 1); plt.imshow(pil_rasm); plt.title("asl", fontsize=9); plt.axis("off")
for i in range(8):
    plt.subplot(1, 9, i + 2)
    plt.imshow(augment(pil_rasm).permute(1, 2, 0))                    # har chaqirishda boshqacha
    plt.axis("off")
plt.suptitle(f"Bitta rasm ({sinflar[pil_javob]}), 8 xil variant")
plt.show()

Chapda asl rasm, o’ngda 8 ta **tasodifiy variant**: ba’zisi ko’zguda, ba’zisi siljigan, ba’zisi qorong’i. Tarmoq har epoxada **boshqacha** variant ko’radi. **Yodlab olish qiyin**, o’rganish kerak. Uzoq o’qitilganda bu **test aniqligini** oshiradi (tarmoq yodlamaydi), qisqa o’qitishda esa natija kechikadi.

**Sinab ko’ring.** `RandomCrop` rasmni siljitadi, `torch.roll` esa aylantiradi (chetdan chiqqan qism boshqa tomondan kiradi). Nega kesishda **`padding=4`** kerak? Avval taxmin qiling, keyin padding’siz `RandomCrop(32)` bilan nima bo’lishini ko’ring.

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 11. CNN qayerda adashadi?

**Hayotda:** o’xshash narsalar odamni ham chalg’itadi: **mushuk va it**, **mashina va yuk mashinasi**. Tarmoq qaysi juftlarda adashadi?

Test to’plamining hammasi bo’yicha **har sinf** uchun aniqlikni hisoblaymiz:

In [ ]:
model_cnn.eval()
barcha_bashorat, barcha_javob = [], []
with torch.no_grad():
    for x, y in test_loader:
        barcha_bashorat.append(model_cnn(x.to(qurilma)).argmax(dim=1).cpu())
        barcha_javob.append(y)
bashorat = torch.cat(barcha_bashorat)
javoblar = torch.cat(barcha_javob)

sinf_aniq = [(bashorat[javoblar == k] == k).float().mean().item() * 100 for k in range(10)]
for nom, a in sorted(zip(sinflar, sinf_aniq), key=lambda t: -t[1]):
    print(f"{nom:<14} {a:5.1f}%")

Jadval **eng oson** (yuqorida: qurbaqa, mashina, yuk mashinasi, kema ~70%) va **eng qiyin** (pastda) sinflarni ko’rsatadi. **Eng qiyin: mushuk (~27%)**, keyin it va kiyik. Hayvonlar bir-biriga o’xshash, mashinalar esa aniq shaklli. **Qaysi juftlarni** adashtiradi?

In [ ]:
from collections import Counter

xato = bashorat != javoblar
juftlar = Counter(zip(javoblar[xato].tolist(), bashorat[xato].tolist()))
for (haqiqiy, taxmin), soni in juftlar.most_common(5):
    print(f"{sinflar[haqiqiy]} ni {sinflar[taxmin]} deb adashgan: {soni} marta")

Eng ko’p xato **hayvonlar orasida**: kiyik, it va mushuk ko’pincha **«qush»** deb adashadi (hammasida tabiiy fon: yashil, ko’k). Yana: **mashina ↔ yuk mashinasi** va **mushuk ↔ it**. Bu **odamlar kabi** xato: tarmoq fon va umumiy shakl bo’yicha o’ylaydi. Chuqurroq tarmoq va ko’proq o’qitish bunday xatolarni kamaytiradi.

**Sinab ko’ring.** Test rasmining qaysi **eng ishonchli xatosi** bor? Ya’ni tarmoq **juda ishonib** adashgan rasm. Avval taxmin qiling: u qanday rasm bo’ladi?

In [ ]:
# Avval taxmin qiling, keyin shu yerga yozib tekshiring

## 12. Siz buni har kuni ishlatasiz

Bugungi CNN — 25 578 ta son, 3 epoxa. Xuddi shu qoida, faqat kattaroq:
- **Telefon galereyasi**: «mushuk» deb qidirsangiz, rasmlarni CNN ko’rib chiqadi;
- **Face ID** va rasmda yuzni topish (kamera yuzga fokus qiladi);
- **Avtomobil kamerasi**: yo’l belgilari, piyodalar, boshqa mashinalar;
- **Tibbiyot**: rentgen va MRT rasmidan kasallikni topishda shifokorga yordam;
- **Qishloq xo’jaligi**: o’simlik bargidagi kasallikni telefon rasmidan aniqlash.

Hammasida **filtrlar**, **pooling**, **o’qitish tsikli**. 1–7 darslarning hammasi shu yerda.

**Keyingi dars (9-dars): transfer learning.** Bugun CNN ni **noldan** o’qitdik: 50 000 rasm, bir necha daqiqa. Millionlab rasmda **oldindan o’qitilgan** katta CNN lar bor (ResNet, MobileNet). Ularni **o’z rasmlaringizga** (masalan, o’zbek taomlari) moslab, **bir necha daqiqada** yaxshi model yasaymiz. Bu 1-oy loyihasi **«Ko’z»** ning boshlanishi.

## Uy ishi

**1. Augmentation bilan o’qiting.** Yuqoridagi `augment` bilan CNN ni o’qiting va **aniqlik qanday o’zgarganini** o’lchang. Quyidagi shablonni ishga tushiring (**5 epoxa**, har ikkisi uchun bir xil; GPU’da taxminan 1–2 daqiqa, GPU’siz ~10 daqiqa). Natijani jadvalga yozing: augmentation’siz va augmentation bilan.

**2. Natijani izohlang (2–3 gap).** Augmentation aniqlikni oshirdimi yoki pasaytirdimi? Nega? Nima uchun augmentation’li tarmoqqa **ko’proq epoxa** kerak bo’lishi mumkin? (Maslahat: tarmoq har epoxada **boshqacha** rasm ko’radi.) Ixtiyoriy: epoxalarni 15 ga oshirib ko’ring: natija o’zgaradimi?

**3. O’zingizning o’zgartirishingiz.** `augment` ga yana bitta o’zgartirish qo’shing (masalan, `transforms.RandomRotation(15)`) va natijani solishtiring.

Boshlash uchun shablon (1-vazifa):

In [ ]:
aug_data = datasets.CIFAR10("data", train=True, download=True, transform=augment)
aug_loader = DataLoader(aug_data, batch_size=128, shuffle=True)

torch.manual_seed(1)
print("--- augmentation'siz ---")
oddiy = o_qit(cnn_yasa(), train_loader, epochlar=5)

torch.manual_seed(1)
print("--- augmentation bilan ---")
aug = o_qit(cnn_yasa(), aug_loader, epochlar=5)

**Javobingiz:** shu katakni ikki marta bosib yozing.

## Chuqurroq (ixtiyoriy): Dropout va chuqurroq tarmoq

1. CNN ga **3-blok** qo’shing: `Conv2d(32, 64, 3, padding=1), ReLU, MaxPool2d(2)`. `Linear` ga kirish soni nechaga o’zgaradi? (Maslahat: shakllar jadvalini qo’lda hisoblang: 32 → 16 → 8 → 4.)
2. `nn.Dropout(0.3)` ni `Linear` dan oldin qo’shing: o’qitishda 30% neyron tasodifiy «uxlatiladi». Aniqlik qanday o’zgardi? Nega overfitting kamayishi mumkin?
3. Uchta tarmoqni (kichik, 3-blokli, Dropout’li) **bitta grafikda** solishtiring.

## Yakun

Bugun siz:
- oddiy tarmoq rasmda **nega** yomon ishlashini ko’rdingiz (qo’shni piksellar yo’qoladi, surilgan rasm boshqa sonlar);
- **filtr** (naqsh qidiruvchi) nima ekanini qo’lda hisoblab, **konvolyutsiyani** o’zingiz yozdingiz;
- **kanal**, **ReLU**, **pooling** va ularning shakllarini tushundingiz;
- o’z **CNN** ingizni yasab, **MLP dan** ancha kam son bilan **yaxshiroq** natijaga erishdingiz;
- tarmoq o’rgangan **filtrlarni** ko’rdingiz;
- **augmentation** va tarmoqning **adashadigan** joylarini ko’rdingiz.

Keyingi darsda (9-dars) **transfer learning**: oldindan o’qitilgan katta CNN ni o’z rasmlaringizga moslab, **«Ko’z»** loyihasini boshlaymiz.